# Wavelet basics, and the checks that the machinery is right

Before anything is measured on XAUUSD, known signals must come back where they were put. This notebook calls the library's transforms and positive controls; it computes nothing itself.

* **The causal transform** is a one-sided MODWT: the level-*j* coefficient at bar *t* is a fixed filter applied to bars *t-L_j+1 .. t* (built from PyWavelets' `db4` filters). No padding is ever needed at the newest bar, and appending bars changes no earlier coefficient.
* **Bands** have physical periods: detail band *j* covers roughly 2^j - 2^(j+1) bars (nominal 1.4 x 2^j for `db4`), which is minutes on 5m bars and hours on 1h bars.
* **White noise is not flat across bands**: band *j* holds 2^-j of its energy, so the *dominant* band is the one that most exceeds its white-noise share.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import polars as pl
from IPython.display import display

from xauusd_quant.features.spectral_config import load_spectral_config
from xauusd_quant.features.wavelet import (
    causal_modwt, circular_modwt, dwt_components, max_level, modwt_filters)
from xauusd_quant.features.wavelet_config import load_wavelet_config
from xauusd_quant.features.wavelet_energy import band_layout, white_noise_shares
from xauusd_quant.research.wavelet_analysis import (
    boundary_study, chirp_control, localized_oscillation_control, multiscale_control)

wavelet = load_wavelet_config(ROOT / "config" / "wavelet.yaml")
spectral = load_spectral_config(ROOT / "config" / "spectral.yaml")

## Filters, levels per window, and the energy decomposition

In [ ]:
f = modwt_filters("db4", 6)
print("filter lengths:", f.lengths)
print("levels per window:", {n: max_level(n, "db4") for n in wavelet.windows})
x = np.random.default_rng(1).normal(size=4096)
w, v = circular_modwt(x, f)
print("energy identity:", (w ** 2).sum() + (v ** 2).sum(), "vs", (x ** 2).sum())

## Bands in physical time (5m and 1h bars, window 512)

In [ ]:
for seconds in (300.0, 3600.0):
    layout = band_layout(512, wavelet="db4", bar_seconds=seconds, min_coefficients=16,
                         fast_slow_boundary_seconds=wavelet.causal_features.fast_slow_boundary_seconds,
                         band_edges_seconds=tuple(wavelet.causal_features.band_edges_seconds))
    display(pl.DataFrame(layout.describe()).with_columns(
        pl.Series("white_noise_share", white_noise_shares(layout))))

## Multiresolution: x = A_J + sum D_j (offline, whole series)

In [ ]:
t = np.arange(1024)
x = np.sin(2 * np.pi * t / 128) + 0.5 * np.sin(2 * np.pi * t / 4)
parts = dwt_components(x, "db4", level=6)
print("max reconstruction error:", np.abs(sum(parts.values()) - x).max())
pl.DataFrame({k: [float((v ** 2).mean())] for k, v in parts.items()})

## Positive controls: a temporary cycle, a chirp, two scales at once

In [ ]:
localized_oscillation_control(wavelet, spectral)

In [ ]:
chirp_control(wavelet, spectral)

In [ ]:
multiscale_control(wavelet)

## The right edge: windowed transforms with padding vs the causal MODWT

The newest coefficient of a *windowed* DWT straddles the edge, so its value depends on how the window is padded. The causal MODWT needs no padding; its cost is a delay of about half a filter length.

In [ ]:
trials, summary = boundary_study(wavelet)
summary